# Comparaison de Modeles ML : Classification du Niveau de Risque des Seismes
Ce notebook compare 3 algorithmes de machine learning pour classer le niveau de risque des seismes selon leur magnitude (`mag`) :

- **Faible (0)** : `mag` < 4.0
- **Moyen (1)** : 4.0 <= `mag` < 6.0
- **Eleve (2)** : `mag` >= 6.0

**Algorithmes evalues :**
1. k-Nearest Neighbors (KNN)
2. Arbre de decision (Decision Tree)
3. Forêt aleatoire (Random Forest)

In [ ]:
# Importation des bibliotheques requises
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
import mlflow
import mlflow.sklearn

# Chargement du dataset
file_path = os.path.join("data", "usgs_earthquakes_2025.csv")
df = pd.read_csv(file_path)
print(f'Taille du jeu de donnees : {df.shape}')

## 1. Preparation de la Cible (Target) et des Caracteristiques (Features)

In [ ]:
# 1. Variable cible (Magnitude)
conditions = [
    df['mag'] < 4.0,
    (df['mag'] >= 4.0) & (df['mag'] < 6.0),
    df['mag'] >= 6.0,
]
choices = [0, 1, 2]  # 0: Faible, 1: Moyen, 2: Eleve
df['risk_level'] = np.select(conditions, choices, default=0)


### Justification du Nettoyage et de l'Exclusion des Variables (Feature Selection)

Afin de construire un modele performant, generalisable et exempt de fuite de donnees (*Data Leakage*), plusieurs variables du dataset ont ete exclues lors du choix des caracteristiques ($X$) :

1. **Variables a valeur constante :**
   - `year` (valeur unique "2025") : Une variable sans variance n'apporte aucune information discriminante pour l'apprentissage et ajoute inutilement de la dimensionnalite.

2. **Identifiants uniques (Haute cardinalite) :**
   - `url`, `detail`, `code`, `ids`, `event_id`, `title`, `place` : Ces champs possèdent 100% (ou presque) de valeurs uniques par enregistrement. Les inclure entraînerait un surapprentissage pur (*overfitting*), car le modèle apprendrait par cœur des identifiants au lieu de repérer des motifs généraux.

3. **Variables entierement ou tres fortement manquantes :**
   - `tz` (100% de valeurs manquantes) : Ne contient aucune donnée exploitable.

4. **Variables d'impact post-evenement (Prevention du Data Leakage) :**
   - `felt`, `cdi`, `mmi`, `alert`, `sig`, `status` : Ces informations représentent l'impact ressenti ou mesuré *après* la survenue du séisme (nombre de signalements citoyens, révisions d'experts, alertes émises). Si l'objectif est d'évaluer le risque dès la détection instrumentale, utiliser ces champs constituerait une fuite de données (*Data Leakage*), faussant artificiellement les performances du modèle en conditions réelles.

5. **Exclusion de la variable source (`mag`) :**
   - `mag` a été retirée des variables prédictives car elle a servi directement à construire la variable cible `risk_level`. Sa conservation donnerait un accès direct à la réponse.

---

**Variables conservees pour la modélisation ($X$) :**
- **Coordonnées géophysiques & spatiales :** `latitude`, `longitude`, `depth`
- **Qualité des mesures physiques de l'épicentre :** `dmin`, `gap`, `rms`, `nst`
- **Contexte technique & catégoriel :** `net`, `magType`, `type`

In [ ]:
# Sélection des variables d'entrée (Features) et de la cible (Target)
features = [
    'latitude',
    'longitude',
    'depth',
    'dmin',
    'gap',
    'rms',
    'nst',
    'net',
    'magType',
    'type',
]

X = df[features].copy()
y = df['risk_level']

# Encodage des colonnes catégorielles
cat_cols = ['net', 'magType', 'type']
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X[cat_cols] = encoder.fit_transform(X[cat_cols])

# Decoupage Train/Test (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Imputation des valeurs manquantes par la médiane (ex. gap, dmin, nst)
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

# Standardisation des données (nécessaire pour KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_scaled = scaler.transform(X_test_imp)

print('Preparation et pretraitement des donnees termines !')

## 2. Entrainement et Evaluation des 3 Modeles

In [ ]:
# Dictionnaire pour stocker les resultats des metriques
results = {}


def evaluate_model(name, y_true, y_pred):
  acc = accuracy_score(y_true, y_pred)
  prec_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
  rec_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
  f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
  f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)

  results[name] = {
      'Accuracy': acc,
      'Precision (Macro)': prec_macro,
      'Recall (Macro)': rec_macro,
      'F1-Score (Macro)': f1_macro,
      'F1-Score (Weighted)': f1_weighted,
  }



mlflow.set_tracking_uri("http://194.238.26.226:5000")
trusted_types = [
    "sklearn.metrics._dist_metrics.EuclideanDistance64",
    "sklearn.neighbors._kd_tree.KDTree",
    "sklearn.tree._tree.Tree"
]

mlflow.end_run()
# ---------------------------------------------------------
# Algorithme 1 : KNN
# ---------------------------------------------------------
experiment_name = "Risk_Zone_KNN_Model_" + str(pd.Timestamp.now().strftime("%Y-%m-%d_%H:%M:%S"))

# 2. Créer l'expérience avec la description
experiment_id = mlflow.create_experiment(
    name=experiment_name,
    tags={
        "mlflow.note.content": "Expérience de détection de zone de risque KNN."
    },
)

# 3. Sélectionner l'expérience pour le suivi
mlflow.set_experiment(experiment_id=experiment_id)

with mlflow.start_run(run_name="KNN_Neighbors_5_Weights_Distance", nested=True):
    knn = KNeighborsClassifier(n_neighbors=5, weights='distance')
    knn.fit(X_train_scaled, y_train)
    y_pred_knn = knn.predict(X_test_scaled)
    evaluate_model('KNN', y_test, y_pred_knn)

mlflow.log_param("n_neighbors", 5)
mlflow.log_param("weights", "distance")

mlflow.log_metric("Accuracy", results['KNN']['Accuracy'])
mlflow.log_metric("Precision", results['KNN']['Precision (Macro)'])
mlflow.log_metric("Recall", results['KNN']['Recall (Macro)'])
mlflow.log_metric("F1-Score-Macro", results['KNN']['F1-Score (Macro)'])
mlflow.log_metric("F1-Score-Weighted", results['KNN']['F1-Score (Weighted)'])
mlflow.sklearn.log_model(sk_model=knn, name="model_KNN",skops_trusted_types=trusted_types)
mlflow.set_tag("mlflow.runName", "KNN_Neighbors_5_Weights_Distance")


# ---------------------------------------------------------
# Algorithme 2 : Decision Tree
# ---------------------------------------------------------
# 1. Définir le nom
experiment_name = "Risk_Zone_DecisionTree_Model_"+ str(pd.Timestamp.now().strftime("%Y-%m-%d_%H:%M:%S"))
# 2. Créer l'expérience avec la description
experiment_id = mlflow.create_experiment(
    name=experiment_name,
    tags={
        "mlflow.note.content": "Expérience de détection de zone de risque Decision Tree."
    },
)

# 3. Sélectionner l'expérience pour le suivi
mlflow.set_experiment(experiment_id=experiment_id)

# S'assurer d'abord qu'aucun run n'est actif
if mlflow.active_run():
    mlflow.end_run()

with mlflow.start_run(run_name="DecisionTree_MaxDepth_10_balanced", nested=True):
    dt = DecisionTreeClassifier(
        max_depth=10, class_weight='balanced', random_state=42
    )
    dt.fit(X_train_imp, y_train)
    y_pred_dt = dt.predict(X_test_imp)
    evaluate_model('Decision Tree', y_test, y_pred_dt)

# Log des métriques et paramètres
mlflow.log_param("max_depth", 10)
mlflow.log_param("class_weight", "balanced")
mlflow.log_param("random_state", 42)

mlflow.log_metric("Accuracy", results['Decision Tree']['Accuracy'])
mlflow.log_metric("Precision", results['Decision Tree']['Precision (Macro)'])
mlflow.log_metric("Recall", results['Decision Tree']['Recall (Macro)'])
mlflow.log_metric("F1-Score-Macro", results['Decision Tree']['F1-Score (Macro)'])
mlflow.log_metric("F1-Score-Weighted", results['Decision Tree']['F1-Score (Weighted)'])
mlflow.sklearn.log_model(sk_model=dt, 
                         name="model_DecisionTree",
                         skops_trusted_types=trusted_types)
mlflow.set_tag("mlflow.runName", "DecisionTree_MaxDepth_10_balanced")


# ---------------------------------------------------------
# Algorithme 3 : Random Forest
# ---------------------------------------------------------

# 1. Définir le nom
experiment_name = "Risk_Zone_RandomForest_Model_"+ str(pd.Timestamp.now().strftime("%Y-%m-%d_%H:%M:%S"))

# 2. Créer l'expérience avec la description
rd_experiment_id = mlflow.create_experiment(
    name=experiment_name,
    tags={
        "mlflow.note.content": "Expérience de détection de zone de risque Random Forest."
    },
)

# 3. Sélectionner l'expérience pour le suivi
mlflow.set_experiment(experiment_id=rd_experiment_id)

# S'assurer d'abord qu'aucun run n'est actif
if mlflow.active_run():
    mlflow.end_run()

with mlflow.start_run(run_name="RandomForest_NEstimators_150_MaxDepth_12_balanced", nested=True):

    rf = RandomForestClassifier(
        n_estimators=150,
        max_depth=12,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1,
    )
rf.fit(X_train_imp, y_train)
y_pred_rf = rf.predict(X_test_imp)
evaluate_model('Random Forest', y_test, y_pred_rf)

# Log des métriques et paramètres
mlflow.log_param("n_estimators", 150)
mlflow.log_param("max_depth", 12)
mlflow.log_param("class_weight", "balanced")
mlflow.log_param("random_state", 42)
mlflow.log_param("n_jobs", -1)

mlflow.log_metric("Accuracy", results['Random Forest']['Accuracy'])
mlflow.log_metric("Precision", results['Random Forest']['Precision (Macro)'])
mlflow.log_metric("Recall", results['Random Forest']['Recall (Macro)'])
mlflow.log_metric("F1-Score-Macro", results['Random Forest']['F1-Score (Macro)'])
mlflow.log_metric("F1-Score-Weighted", results['Random Forest']['F1-Score (Weighted)'])
mlflow.sklearn.log_model(sk_model=rf, 
                         name="model_RandomForest",
                         skops_trusted_types=trusted_types)
mlflow.set_tag("mlflow.runName", "RandomForest_NEstimators_150_MaxDepth_12_balanced")


print('Tous les modeles ont été entraines et evalues.')

## 3. Rapport de Classification Detailled (Print par Modèle)

In [ ]:
target_names = ['Faible (<4.0)', 'Moyen (4.0-5.9)', 'Élevé (>=6.0)']

print('=================== 1. KNN ===================')
print(classification_report(y_test, y_pred_knn, target_names=target_names))

print('=================== 2. DECISION TREE ===================')
print(classification_report(y_test, y_pred_dt, target_names=target_names))

print('=================== 3. RANDOM FOREST ===================')
print(classification_report(y_test, y_pred_rf, target_names=target_names))

## 4. Tableau Comparatif des Performance Metrics

In [ ]:
df_results = pd.DataFrame(results).T
df_results = df_results.round(4)
df_results.sort_values(by='F1-Score (Macro)', ascending=False)

## 5. Analyse Comparative et Choix du Meilleur Modele

### Pourquoi se fier au **F1-Score (Macro)** plutôt qu'à l'Accuracy ?
Le dataset présente un fort déséquilibre de classe (la majorité des séismes sont mineurs, alors que les séismes majeurs de classe `Élevé` sont rares). L'**Accuracy** peut donner une illusion de performance en prédisant bien les classes majoritaires tout en ignorant la classe critique `Élevé`. Le **F1-Score Macro** évalue chaque classe avec la même importance, garantissant ainsi que le modèle détecte efficacement les séismes graves.

---

### Analyse des Algorithmes :

1. **KNN (k-Nearest Neighbors) :**
   - **Avantage :** Modèle simple et intuitif.
   - **Inconvénient :** Très sensible aux données manquantes et à l'échelle des variables. Temps d'inférence plus élevé sur de grands jeux de données. Il tend à favoriser la classe majoritaire aux dépens de la classe rare `Élevé`.

2. **Decision Tree (Arbre de Décision) :**
   - **Avantage :** Facile à interpréter et gère bien le déséquilibre grâce à `class_weight='balanced'`.
   - **Inconvénient :** Sujet au surapprentissage (*overfitting*), ce qui réduit sa capacité de généralisation sur des séismes non vus lors de l'entraînement.

3. **Random Forest (Forêt Aléatoire) — 🏆 MEILLEUR CHOIX :**
   - **Avantage :** Modèle d'ensemble (*bagging*) réduisant significativement la variance par rapport à un simple arbre de décision.
   - **Performance :** Il obtient généralement le meilleur compromis entre **Recall (Rappel)** sur la classe `Élevé` et **Précision globale**, en gérant naturellement les corrélations spatiales (`latitude`, `longitude`, `depth`) et les caractéristiques techniques des stations (`gap`, `dmin`, `rms`).

---

### **Conclusion / Recommandation :**
Le **Random Forest** est retenu comme le **meilleur modèle**. Il offre la meilleure capacité de généralisation et minimise le risque d'omettre un séisme à risque élevé (faibles faux négatifs sur la classe critique).

In [ ]:
# Sauvegarder le meilleur modèle

# Sauvegarder le meilleur modèle selon le F1-Score Macro
best_model_name = df_results['F1-Score (Macro)'].idxmax()

models = {
    'KNN': knn,
    'Decision Tree': dt,
    'Random Forest': rf,
}

best_model = models[best_model_name]

if mlflow.active_run():
    mlflow.end_run()

mlflow.set_experiment(experiment_id=rd_experiment_id)

with mlflow.start_run(run_name=f"Champion_{best_model_name}"):
    mlflow.set_tag("is_champion", "true")
    mlflow.set_tag(
        "mlflow.note.content",
        "Meilleur modèle basé sur le F1-Score (Macro)"
    )
    mlflow.set_tag("model_name", best_model_name)

    mlflow.log_metric(
        "F1-Score-Macro",
        results[best_model_name]["F1-Score (Macro)"]
    )

    # Convertir le DataFrame en Dataset MLflow
    dataset = mlflow.data.from_pandas(
        df,
        source="../data/usgs_earthquakes_2025.csv",  # Chemin source ou URL du dataset
        name="risk_zone_dataset"            # Nom affiché dans MLflow
    )

    #  Enregistrer le dataset dans le Run
    mlflow.log_input(dataset, context="training")



    mlflow.sklearn.log_model(
        best_model,
        name="champion_model",
        registered_model_name="Risk_Zone_RandomForest_Model",  # Nom du modèle dans le Model Registry
        skops_trusted_types=trusted_types
    )

    mlflow.set_experiment_tag(key="is_champion", value="true")
    mlflow.set_tag("mlflow.note.content", "Meilleur modèle basé sur le F1-Score (Macro)")



output_dir = "../models"
os.makedirs(output_dir, exist_ok=True)

# Sauvegarde directe dans le dossier models
joblib.dump(best_model, os.path.join(output_dir, "Risk_Zone_RandomForest_champion_model.joblib"))

print(f"Modèle sauvegardé avec succès : {best_model_name}")
